In [1]:
import xarray as xr

ds_sample = xr.open_zarr("../data/SV_heatmap_2025_20250101.zarr")
print(ds_sample)

/home/b/b384140/.conda/envs/parcels/lib/python3.14/site-packages/pyproj/network.py:59: UserWarning: pyproj unable to set PROJ database path.
  _set_context_ca_bundle_path(ca_bundle_path)


<xarray.Dataset> Size: 2GB
Dimensions:     (trajectory: 230656, obs: 240)
Coordinates:
  * obs         (obs) int32 960B 0 1 2 3 4 5 6 7 ... 233 234 235 236 237 238 239
  * trajectory  (trajectory) int64 2MB 0 1 2 3 4 ... 230652 230653 230654 230655
Data variables:
    lat         (trajectory, obs) float32 221MB dask.array<chunksize=(230656, 1), meta=np.ndarray>
    lon         (trajectory, obs) float32 221MB dask.array<chunksize=(230656, 1), meta=np.ndarray>
    pid_orig    (trajectory, obs) float64 443MB dask.array<chunksize=(230656, 1), meta=np.ndarray>
    time        (trajectory, obs) datetime64[ns] 443MB dask.array<chunksize=(230656, 1), meta=np.ndarray>
    z           (trajectory, obs) float32 221MB dask.array<chunksize=(230656, 1), meta=np.ndarray>
Attributes:
    Conventions:            CF-1.6/CF-1.7
    feature_type:           trajectory
    ncei_template_version:  NCEI_NetCDF_Trajectory_Template_v2.0
    parcels_kernels:        SVParticleAdvectionRK4CheckOutOfBoundsCheckErro

In [2]:
import glob
import re
from datetime import datetime

files = sorted(glob.glob("../data/SV_heatmap_2025_*.zarr"))

def extract_date(fname):
    match = re.search(r"(\d{8})\.zarr$", fname)
    return datetime.strptime(match.group(1), "%Y%m%d")

files_with_dates = [(f, extract_date(f)) for f in files]
files_with_dates.sort(key=lambda x: x[1])

print(len(files_with_dates), "files found")
print(files_with_dates[:3], "...", files_with_dates[-3:])

73 files found
[('SV_heatmap_2025_20250101.zarr', datetime.datetime(2025, 1, 1, 0, 0)), ('SV_heatmap_2025_20250106.zarr', datetime.datetime(2025, 1, 6, 0, 0)), ('SV_heatmap_2025_20250111.zarr', datetime.datetime(2025, 1, 11, 0, 0))] ... [('SV_heatmap_2025_20251217.zarr', datetime.datetime(2025, 12, 17, 0, 0)), ('SV_heatmap_2025_20251222.zarr', datetime.datetime(2025, 12, 22, 0, 0)), ('SV_heatmap_2025_20251227.zarr', datetime.datetime(2025, 12, 27, 0, 0))]


In [3]:
datasets = []
for f, date in files_with_dates:
    ds = xr.open_zarr(f)
    ds = ds.expand_dims(release_date=[date])
    datasets.append(ds)

combined = xr.concat(datasets, dim="release_date")

In [4]:
combined = combined.chunk({"release_date": 1})  # or {"time": -1} etc, adjust to your dims

combined.to_zarr(
    "SV_heatmap_2025.zarr",
    mode="w",
    consolidated=True
)